<div dir="rtl" style="direction:rtl;text-align:right;font:700 19px Tahoma,Arial,sans-serif;line-height:1.9">
# KhaledDSML · 0.7 · مختبر Spark وDask

## تحليل البيانات الضخمة وتعلّم الآلة الموزع — جامعة الوادي

برنامج عملي يشرح قراءة البيانات، وفحصها، والتقسيم الزمني، وتعلّم الآلة باستخدام PySpark. النتائج تُعرض بأسلوب **Analytics Studio** بالألوان المتدرجة، بالعربية من اليمين والإنجليزية من اليسار.

**هذه نسخة تطويرية**؛ الإصدار الرسمي الحالي على PyPI هو 0.6.0، ونتائج هذا الدفتر تعليمية وليست بحثًا منشورًا.
</div>

<div dir="rtl" style="text-align:right">
## 1. تثبيت PySpark وDask مع نسخة التطوير

شغّل الخلية ثم اختر Runtime → Restart session. تحتاج Spark إلى Java متوافق على الجهاز أو Colab.
</div>

In [ ]:
%pip install --upgrade "khaled-dsml[distributed]==0.6.0"
%pip install --no-deps --force-reinstall --no-cache-dir "git+https://github.com/noble2013/KhaledDSML.git@feature/v0.7-distributed-analytics"

<div dir="rtl" style="text-align:right">
## 2. قراءة Spark DataFrame وفحص القيم المفقودة

تُحسب جميع مقاييس الجودة داخل Spark؛ لا تُنقل إلى pandas إلا جداول الملخصات الصغيرة.
</div>

In [ ]:
import numpy as np
import pandas as pd
import khaled_dsml as k
from khaled_dsml.distributed import create_spark_session
from IPython.display import display

spark=create_spark_session(master="local[2]")
spark.sparkContext.setLogLevel("ERROR")
data=spark.createDataFrame(
    [("الوادي",100., "2026-01-04","جيد"),
     ("الوادي",100., "2026-01-04","جيد"),
     ("الجزائر",200., "2026-02-04",""),
     ("بسكرة",None, "2026-03-04",None)],
    ["wilaya","montant_da","date","note"]
)
quality=k.spark_missingness_detailed(data)
profile=k.spark_numeric_profile(data,columns=["montant_da"])
duplicates=k.spark_duplicate_summary(data)

display(quality,profile)
k.display_analysis_dashboard(
    duplicates,
    title_ar="تقرير تدقيق البيانات الضخمة",
    title_en="SPARK · DATA QUALITY",
    theme="aurora",font_size=22,
)

<div dir="rtl" style="text-align:right">
## 3. الفصل الزمني الآمن للبيانات

نُبقي التواريخ غير الصالحة في مجموعة مستقلة بدل إسقاطها بصمت.
</div>

In [ ]:
parts=k.spark_time_holdout(data,date_column="date",cutoff="2026-02-01")
print("Training:",parts["train"].count())
print("Testing:",parts["test"].count())
print("Invalid dates:",parts["invalid"].count())

<div dir="rtl" style="text-align:right">
## 4. نموذج لوجستي كامل على Spark ML

تُدرّب خطّ المعالجة على بيانات التدريب فقط، ثم يُقيّم على بيانات لم تدخل التدريب. المثال اصطناعي لتوضيح الأداة ولا يثبت دقة نموذج بحثي.
</div>

In [ ]:
rng=np.random.default_rng(7)
rows=[]
for i in range(240):
    x=float(rng.normal())
    second=float(rng.normal())
    label=float(x+second*.2>0)
    rows.append((i,x,second,label))
full=spark.createDataFrame(rows,["id","economic_feature","market_feature","label"])
train=full.where("id < 180")
test=full.where("id >= 180")

model=k.spark_fit_logistic_classifier(
    train,feature_columns=["economic_feature","market_feature"],
    max_iter=45,reg_param=.01
)
predictions=model.transform(test)
validation=k.spark_binary_classification_metrics(predictions)
k.display_analysis_dashboard(
    validation,title_ar="تقييم النموذج الموزع",
    title_en="SPARK ML · HELD-OUT MODEL VALIDATION",
    theme="ocean",font_size=21,
)

<div dir="rtl" style="text-align:right">
## 5. مقارنة النتائج على Dask

البيانات الأصلية في Dask تُقرأ كسولًا وتُختزل إلى جدول إحصائي صغير.
</div>

In [ ]:
import dask.dataframe as dd

dask_df=dd.from_pandas(pd.DataFrame({
    "wilaya":["الوادي","الوادي","بسكرة","الجزائر"],
    "montant_da":[100.,100.,150.,200.],
    "date":["2026-01-04","2026-01-04","2026-02-07","bad"],
}),npartitions=2)
display(k.dask_numeric_profile(dask_df))
dask_dup=k.dask_duplicate_summary(dask_df)
k.display_analysis_dashboard(
    dask_dup,title_ar="مؤشرات تكرار السجلات",
    title_en="DASK · DUPLICATE ANALYSIS",
    theme="amethyst",font_size=22,
)

<div dir="rtl" style="text-align:right">
## 6. حفظ التقرير الجامعي

تحويل المقاييس المجمّعة إلى HTML أنيق قابل للطباعة إلى PDF.
</div>

In [ ]:
path=k.save_analysis_dashboard(
    [duplicates,validation,dask_dup],
    "KhaledDSML_Spark_Dask_Analytics.html",
    title_ar="مرصد علم البيانات والتحليل الموزع",
    title_en="SPARK & DASK · ACADEMIC ANALYTICS",
    theme="aurora",font_size=21
)
print("Report saved:",path)

<div dir="rtl" style="direction:rtl;text-align:right;font:600 16px Tahoma,Arial,sans-serif;line-height:1.8">
## ملاحظات مهمة

- PySpark إضافة اختيارية وتحتاج Java.
- لا تستخدم مؤشرات التدريب لادعاء جودة التعميم؛ التقييم يحتاج اختبارًا مستقلًا دون تسرب.
- لا تَعِد هذه الوظائف بعمل غير محدود على أي حاسوب؛ تعتمد الحدود على حجم الذاكرة والإعدادات والموارد.
- لا يوجد نقل لسجلات البيانات الأصلية إلى pandas في الملخصات الجديدة.
- هذا الدفتر يقدم جزءًا من كتالوج الاختبارات، وليس استكمالًا لكل بنوده الـ256.
</div>